<a href="https://colab.research.google.com/github/NK-cloud-hub/testrepo/blob/main/ML_detection_of_anomalous_energy_consumption.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np

from statsmodels.tsa.seasonal import STL
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [4]:
def load_machine(path, machine_id):
    df = pd.read_csv(path)
    df["machine_id"] = machine_id
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True).dt.tz_convert(None)
    return df

m1 = load_machine("df_recipe_performance_machine1.csv", "M1")
m2 = load_machine("df_recipe_performance_machine2.csv", "M2")
m3 = load_machine("df_recipe_performance_machine3.csv", "M3")

df = pd.concat([m1, m2, m3], ignore_index=True).sort_values("timestamp")

In [34]:
# 2. Energy-Intensity Proxy
# -----------------------
df["processed"] = df["processed"].replace(0, np.nan).fillna(1)  # guard div/0
df["actual_time_per_bottle"] = (
    df["runningtime"] + df["equipmentdowntime"] + df["externaltime"]
) / df["processed"]

df["expected_time_per_bottle"] = 1.0 / np.maximum(df["designspeed"], 1e-6)
df["EII"] = df["actual_time_per_bottle"] / df["expected_time_per_bottle"]
df["sec_per_1k"] = df["actual_time_per_bottle"] * 1000.0


In [35]:
# 3. Rolling baseline + residuals
# -----------------------
def rolling_residuals(g, value_col="EII", window=24):
    g = g.sort_values("timestamp").set_index("timestamp").asfreq("H")
    series = g[value_col].interpolate(limit_direction="both")

    out = g.copy()
    out["trend"] = series.rolling(window, min_periods=1).median()
    out["resid"] = series - out["trend"]

    resid = out["resid"].values
    med = np.nanmedian(resid)
    mad = np.nanmedian(np.abs(resid - med)) + 1e-6
    out["resid_z_robust"] = (out["resid"] - med) / mad
    return out.reset_index()

parts = []
for (mid, rec), g in df.groupby(["machine_id", "recipe"], dropna=False):
    sub = g[["timestamp", "machine_id", "recipe", "EII"]].copy()
    r = rolling_residuals(sub, value_col="EII", window=24)
    r["machine_id"] = mid
    r["recipe"] = rec
    parts.append(r)

df_resid = pd.concat(parts, ignore_index=True)

/tmp/ipython-input-1531803268.py:4: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  g = g.sort_values("timestamp").set_index("timestamp").asfreq("H")
/tmp/ipython-input-1531803268.py:4: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  g = g.sort_values("timestamp").set_index("timestamp").asfreq("H")
/tmp/ipython-input-1531803268.py:4: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  g = g.sort_values("timestamp").set_index("timestamp").asfreq("H")
/tmp/ipython-input-1531803268.py:4: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  g = g.sort_values("timestamp").set_index("timestamp").asfreq("H")
/tmp/ipython-input-1531803268.py:4: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  g = g.sort_values("timestamp").set_index("timestamp").as

In [36]:
# Merge back
df = df.merge(
    df_resid[["timestamp", "machine_id", "recipe", "trend", "resid", "resid_z_robust"]],
    on=["timestamp", "machine_id", "recipe"],
    how="left",
)

In [37]:
# Flag anomalies (statistical)
df["anomaly_stat"] = (df["resid_z_robust"].abs() > 3.5).astype(int)

In [38]:
# 4. Isolation Forest (context-aware)
# -----------------------
df["downtime_ratio"] = (
    df["equipmentdowntime"] / np.maximum(df["runningtime"] + df["equipmentdowntime"], 1)
).clip(0, 1)
df["throughput_kbps"] = df["processed"] / np.maximum(df["runningtime"], 1)

num_features = [
    "EII", "sec_per_1k", "efficiency", "availability", "oee",
    "avgprocessspeed", "maxprocessspeed", "minprocessspeed",
    "downtime_ratio", "throughput_kbps", "resid"
]
cat_features = ["machine_id", "recipe"]

X = df[num_features + cat_features].copy()
X[num_features] = X[num_features].replace([np.inf, -np.inf], np.nan).fillna(0)

pre = ColumnTransformer([
    ("num", "passthrough", num_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features)
])

iso = IsolationForest(
    n_estimators=300,
    contamination=0.02,  # flag ~2% as anomalies
    random_state=42
)

pipe = Pipeline([("pre", pre), ("iso", iso)])
pipe.fit(X)

scores = pipe.named_steps["iso"].score_samples(pipe.named_steps["pre"].transform(X))
df["if_score"] = scores
if_thr = np.percentile(scores, 2)  # worst 2%
df["anomaly_if"] = (scores <= if_thr).astype(int)

In [39]:
# 5. Final anomaly flag
# -----------------------
df["anomaly_final"] = ((df["anomaly_stat"] == 1) | (df["anomaly_if"] == 1)).astype(int)

In [40]:
# 6. Outputs for report
# -----------------------
anomalies = df.loc[df["anomaly_final"] == 1, [
    "timestamp", "machine_id", "recipe", "EII", "sec_per_1k",
    "efficiency", "availability", "oee", "downtime_ratio",
    "processed", "runningtime", "equipmentdowntime",
    "resid_z_robust", "if_score", "anomaly_stat", "anomaly_if"
]]

pareto_recipe = anomalies.groupby(["machine_id", "recipe"]).size().sort_values(ascending=False)
daily_counts = df.groupby([df["timestamp"].dt.date, "machine_id"])["anomaly_final"].sum()

print("🔎 Top anomalous recipes:\n", pareto_recipe.head(10), "\n")
print("📊 Daily anomaly counts (head):\n", daily_counts.head(20), "\n")


🔎 Top anomalous recipes:
 machine_id  recipe    
M1          recipe_3      631
M3          recipe_3      629
            recipe_4      522
            recipe_2      353
M1          recipe_11     300
            recipe_133    284
M2          recipe_3      267
M3          recipe_50     264
            recipe_17     252
            recipe_60     252
dtype: int64 

📊 Daily anomaly counts (head):
 timestamp   machine_id
2024-01-24  M3             6
2024-01-25  M2            12
            M3            20
2024-01-26  M2            24
            M3             1
2024-01-27  M2            23
            M3             0
2024-01-28  M2            24
2024-01-29  M1             4
            M2            23
            M3             8
2024-01-30  M1             7
            M2            19
            M3             1
2024-01-31  M1            12
            M2            24
            M3            12
2024-02-01  M1             4
            M2            24
            M3            22
N